### Imports

In [3]:
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from Springer_Segmentation.gerar_modelo import treinar_modelo_springer
from source.extrair_features import extrair_features

# Segmentação de Springer
O algoritmo proposto por Springer et al. (2016) é o estado da arte para a segmentação de sinais de fonocardiograma (PCG). Utilizando Modelos Ocultos de Markov com Duração Explícita (HSMM) e o algoritmo de Viterbi, o modelo analisa características do áudio (como Transformadas de Wavelet e Energia de Shannon) para delinear o ciclo cardíaco com precisão fisiológica.

Sua função neste projeto é receber o áudio bruto e classificar o sinal em quatro fases mecânicas fundamentais:

- S1 (Primeira Bulha): Início da sístole (fechamento das válvulas mitral e tricúspide).

- Sístole: Período de ejeção ventricular.

- S2 (Segunda Bulha): Início da diástole (fechamento das válvulas aórtica e pulmonar).

- Diástole: Período de relaxamento e enchimento.

A partir dessa marcação temporal confiável, torna-se possível extrair as métricas de duração e amplitude que alimentarão os classificadores de Machine Learning nas etapas seguintes.

## Gerando o Modelo de Segmentação de Springer

In [ ]:
# Configure os caminhos do seu computador
CAMINHO_TCC = r"C:\Users\gusta\TCC\TCC"
PASTA_REPO = os.path.join(CAMINHO_TCC, "Springer_Segmentation")

# A pasta de treino do Physionet e onde o modelo vai nascer
PASTA_PHYSIONET = os.path.join(PASTA_REPO, "training_data")
MODELO_PKL = os.path.join(CAMINHO_TCC, "springer_segmentation_model.pkl")

modelo_final = treinar_modelo_springer(
    caminho_repo = PASTA_REPO,
    pasta_dados = PASTA_PHYSIONET,
    arquivo_saida = MODELO_PKL,
    max_audios = 1000
)

## Extraindo as features

In [ ]:
CAMINHO_TCC = r"C:\Users\gusta\TCC\TCC"
PASTA_REPO = os.path.join(CAMINHO_TCC, "Springer_Segmentation")
MODELO_PKL = r"springer_segmentation_model.pkl"
PASTA_AUDIOS = r"C:\Users\gusta\TCC\TCC\data\train"
ARQUIVO_SAIDA = os.path.join(CAMINHO_TCC, "features_extraidas.csv")

features_extraidas = extrair_features(
    caminho_repo = PASTA_REPO,
    caminho_modelo = MODELO_PKL,
    pasta_audios = PASTA_AUDIOS,
    arquivo_saida = ARQUIVO_SAIDA
)

features_extraidas.head()

### 🔗 1. Fusão de Dados (Data Fusion)
Nesta etapa, unificamos as características temporais extraídas dos áudios com os rótulos clínicos e metadados dos pacientes. Como o arquivo original de treino possui os áudios dispostos em colunas (formato *wide*), utilizamos a técnica de *Melt* (unpivot) para transformar cada gravação em uma linha individual (formato *long*), permitindo o cruzamento exato com as *features* matemáticas.

In [7]:
caminho_features = 'features_extraidas.csv'
caminho_train = r'data\train.csv'
caminho_meta = r'data\additional_metadata.csv'

# 1. Carregar as tabelas
df_features = pd.read_csv(caminho_features)
df_train = pd.read_csv(caminho_train)
df_meta = pd.read_csv(caminho_meta)

# 2. Reorganizar o Train.csv (Melt) e preparar as chaves de cruzamento
colunas_audios = [f'recording_{i}' for i in range(1, 9)]
df_train_linhas = df_train.melt(
    id_vars=['patient_id', 'AS', 'AR', 'MR', 'MS', 'N'], 
    value_vars=colunas_audios, 
    value_name='nome_do_audio_sem_wav'
).dropna(subset=['nome_do_audio_sem_wav']) # Já remove os vazios na mesma linha

df_train_linhas['arquivo_wav'] = df_train_linhas['nome_do_audio_sem_wav'].astype(str) + '.wav'
df_features = df_features.rename(columns={'patient_id': 'arquivo_wav'})

# 3. A Grande Fusão (Merge)
df_rotulos_completos = pd.merge(df_train_linhas, df_meta, on='patient_id', how='left')
dataset_final = pd.merge(df_features, df_rotulos_completos, on='arquivo_wav', how='inner')

# Limpar colunas temporárias que não precisamos mais
dataset_final = dataset_final.drop(columns=['variable', 'nome_do_audio_sem_wav'])

# --- INÍCIO DAS TRANSFORMAÇÕES NUMÉRICAS (No mesmo DataFrame) ---

# 4. Encoding (Transformar em números)
dataset_final['Gender'] = dataset_final['Gender'].map({'M': 1, 'F': 0})
dataset_final['Lives'] = dataset_final['Lives'].map({'U': 1, 'R': 0})

# 5. Converter Idade de Texto para Número Contínuo
def converter_idade(idade_str):
    if pd.isna(idade_str): return np.nan
    try:
        partes = str(idade_str).split('-')
        return (int(partes[0]) + int(partes[1])) / 2.0
    except:
        return np.nan 

dataset_final['Age'] = dataset_final['Age'].apply(converter_idade)

# 6. Preencher valores vazios com a média
colunas_para_preencher = ['Age', 'Gender', 'Smoker', 'Lives']
for col in colunas_para_preencher:
    dataset_final[col] = dataset_final[col].fillna(dataset_final[col].mean())

# 7. Reorganizar as colunas para ficar elegante
colunas_organizadas = ['patient_id', 'arquivo_wav', 'AS', 'AR', 'MR', 'MS', 'N', 
                       'Age', 'Gender', 'Smoker', 'Lives'] + [col for col in df_features.columns if col != 'arquivo_wav']
dataset_final = dataset_final[colunas_organizadas]

dataset_final = dataset_final.dropna(how='any')

caminho_salvar = 'dataset_final.csv'
dataset_final.to_csv(caminho_salvar, index=False)

print(f"Dimensões finais: {dataset_final.shape}")

# Exibe as primeiras linhas direto no Notebook
dataset_final.head()

Dimensões finais: (864, 31)


,patient_id,arquivo_wav,AS,AR,MR,MS,N,Age,Gender,Smoker,...,m_Ratio_SysRR,sd_Ratio_SysRR,m_Ratio_DiaRR,sd_Ratio_DiaRR,m_Ratio_SysDia,sd_Ratio_SysDia,m_Amp_SysS1,sd_Amp_SysS1,m_Amp_DiaS2,sd_Amp_DiaS2
0,patient_016,AR_016_sit_Aor.wav,0.0,1.0,0.0,0.0,0.0,34.5,1,1,...,0.091438,0.019738,0.660700,0.034412,0.140255,0.039258,0.412363,0.123084,0.884418,0.351987
1,patient_016,AR_016_sit_Mit.wav,0.0,1.0,0.0,0.0,0.0,34.5,1,1,...,0.107745,0.023489,0.641216,0.037205,0.169973,0.042578,0.391649,0.132505,0.898236,0.316727
2,patient_016,AR_016_sit_Pul.wav,0.0,1.0,0.0,0.0,0.0,34.5,1,1,...,0.222610,0.007361,0.526831,0.016333,0.423049,0.021189,0.201668,0.049161,0.787470,0.258047
3,patient_016,AR_016_sit_Tri.wav,0.0,1.0,0.0,0.0,0.0,34.5,1,1,...,0.217510,0.014600,0.539695,0.022940,0.404594,0.042584,0.279554,0.205512,0.716258,0.249561
4,patient_016,AR_016_sup_Aor.wav,0.0,1.0,0.0,0.0,0.0,34.5,1,1,...,0.042312,0.008571,0.704802,0.053228,0.061668,0.022319,0.357486,0.267772,0.834465,0.384031


### Seleção de Modelos (Benchmarking)
Nesta etapa, treinamos e avaliamos múltiplos algoritmos de *Machine Learning* na sua configuração padrão (*baseline*). O objetivo é identificar qual arquitetura consegue capturar melhor os padrões das características extraídas (Wavelets e durações de Liu et al.) para prever a presença de **Estenose Aórtica (AS)**.

Para garantir uma comparação justa entre modelos baseados em árvores e modelos lineares/distância, os dados preditores (X) foram normalizados utilizando o `StandardScaler`.

**Modelos Testados:**
1. Random Forest (Floresta Aleatória)
2. Gradient Boosting (Árvores em Cascata)
3. Regressão Logística (Estatística Clássica)
4. Support Vector Machine - SVM (Fronteiras de Decisão)
5. K-Nearest Neighbors - KNN (Agrupamento por Similaridade)

In [9]:
# 1. Carregar os Dados
caminho_dataset = 'dataset_final.csv'
df = pd.read_csv(caminho_dataset)

# 2. Separar Preditores (X) e o Alvo (y)
# Mantemos o foco na Estenose Aórtica (AS) para este teste
colunas_para_remover = ['patient_id', 'arquivo_wav', 'AS', 'AR', 'MR', 'MS', 'N']
X = df.drop(columns=colunas_para_remover)
y = df['AS'] 

# 3. Dividir em Treino (80%) e Teste/Prova (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 4. Padronização (Deixa a competição justa para o SVM, KNN e Regressão Logística)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. O "Grid de Largada": Nossos 5 competidores
modelos = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Regressão Logística": LogisticRegression(max_iter=1000, random_state=42),
    "SVM (Máquina de Vetores)": SVC(random_state=42),
    "KNN (Vizinhos Próximos)": KNeighborsClassifier()
}

# 6. A Corrida
resultados = []

for nome_modelo, algoritmo in modelos.items():
    # Treina a IA
    algoritmo.fit(X_train_scaled, y_train)
    
    # Aplica a prova final
    y_pred = algoritmo.predict(X_test_scaled)
    
    # Calcula as notas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # Guarda o resultado
    resultados.append({
        "Modelo": nome_modelo,
        "Acurácia": round(acc * 100, 2),
        "Precisão (Falsos Alarmes)": round(prec * 100, 2),
        "Recall (Acha os Doentes)": round(rec * 100, 2),
        "F1-Score (A Média de Ouro)": round(f1 * 100, 2)
    })

# 7. O Pódio
df_resultados = pd.DataFrame(resultados)
# Ordenamos do melhor para o pior usando a nossa métrica mais importante (F1-Score)
df_resultados = df_resultados.sort_values(by="F1-Score (A Média de Ouro)", ascending=False).reset_index(drop=True)

# Aplica uma corzinha no Jupyter para destacar os valores altos
df_resultados.style.background_gradient(cmap='Blues')

,Modelo,Acurácia,Precisão (Falsos Alarmes),Recall (Acha os Doentes),F1-Score (A Média de Ouro)
0,Gradient Boosting,85.550000,86.960000,67.800000,76.190000
1,KNN (Vizinhos Próximos),84.390000,82.000000,69.490000,75.230000
2,Random Forest,84.390000,86.360000,64.410000,73.790000
3,SVM (Máquina de Vetores),83.820000,87.800000,61.020000,72.000000
4,Regressão Logística,80.920000,78.260000,61.020000,68.570000
